# Stage 2: the linearization as a `LinearOperator`

A `LinearizationMethod` names the linear operator that represents a nonlinear flow. `lin_ode`
provides three. They give the same expectation value
([arXiv:2202.02188](https://arxiv.org/abs/2202.02188)), so the choice is about which object carries
the time dependence, and therefore about what you supply and what you read out.

| method | you evolve | you keep fixed | operator |
|---|---|---|---|
| `Koopman` | the observable $g(u)$ | the density | $\mathcal K=\sum_s f_s\,\partial_s$ |
| `Liouville` | the density $\rho(u)$ | the observable | $\mathcal L=-\sum_s \partial_s\,f_s$ |
| `KoopmanVonNeumann` | the wavefunction $\psi$, with $\rho=\psi^{*}\psi$ | the observable, as a matrix | $\mathcal L_{\rm KvN}=-\tfrac12\sum_s\bigl(\partial_s f_s+f_s\partial_s\bigr)$ |

A KvN generator is anti-Hermitian, so its solution oscillates instead of growing
([`kvn_heisenberg`](../examples/kvn_heisenberg.ipynb) shows it).

The `LinearOperator` you get back is basis-free. There is no truncation, no matrix, and the grid
size is still symbolic. A basis enters only at the matrix stage ([`05_matrix`](05_matrix.ipynb)).

In [1]:
import sympy as sp

from lin_ode import OrdinaryDifferentialSystem
from lin_ode.operators import Koopman, KoopmanVonNeumann, Liouville

u, i = sp.IndexedBase("u"), sp.Idx("i", (0, 3))
ode = OrdinaryDifferentialSystem.from_sympy(u[i] * (1 - u[i]), u, (i,))   # logistic reaction

for method in (Koopman(), Liouville(), KoopmanVonNeumann()):
    operator = method.create_operator(ode)
    print(f"{type(method).__name__:19} diff_base={operator.diff_base}   {operator.operator.expr}")

Koopman             diff_base=K   (1 - U[i])*U[i]*K[i]
Liouville           diff_base=K   -K[i]*(1 - U[i])*U[i]
KoopmanVonNeumann   diff_base=K   -((1 - U[i])*U[i]*K[i] + K[i]*(1 - U[i])*U[i])/2


- `method.create_operator(ode)` creates a LinearOperator. It is an untruncated, basis-independent intermediate symbol represented by `U[i]` and `K[i]`. Here $K_i\equiv\partial/\partial u_i$ and $U[i] = u_i$.
- `diff_base` in the output above is the placeholder for differentiation.

The LinearOperator is a symbol constant on RAM, no matter how many sites you have in the system. A basis enters only at the matrix stage ([`05_matrix`](05_matrix.ipynb)).

## Liouville and KvN need a square-integrable basis

An expectation value is an integral, so the basis you take it in must have an inner product. That
requires its functions to be square-integrable. `GaussianHermiteBasis` satisfies that, and
`OperatorMatrix` raises `IncompatibleBasisError` on Carleman basis
([`04_basis`](04_basis/index.ipynb)).

Being square-integrable is what makes the expansion defined, not what makes it exact. On
`GaussianHermiteBasis` a Liouville matrix is built, but at a finite `degree` the truncation drops
part of the density, so $\tfrac{d}{dt}\int\rho\,du$ is not exactly zero.

One `LinearOperator` serves every basis, and nothing here has fixed one yet.
[`04_basis`](04_basis/index.ipynb) chooses one, and [`05_matrix`](05_matrix.ipynb) projects the
operator onto it.

---

[← 1. The ODE](02_ode.ipynb) · [Notebook index](../README.md) · [3. Bases →](04_basis/index.ipynb)